## Data Ingestion


In [19]:
from langchain_community.document_loaders import TextLoader

txt_loader = TextLoader(file_path='Daimler.txt',encoding='utf-8')
docs = txt_loader.load()
docs

[Document(metadata={'source': 'Daimler.txt'}, page_content='Dealer Onboarding and Financial Processes at Daimler\n\nDaimler utilizes a structured and systematic approach for onboarding dealers through the Dealer Profile module. This module serves as the initial entry point for registering new dealers into the system and capturing all relevant information required for further processing and verification.\n\nWithin the Dealer Profile, detailed information about the dealer is collected. This includes personal details such as first name and last name, as well as business-related attributes like the type of franchise and line of business. In addition, geographic information such as country and region is recorded to ensure proper classification and compliance with regional regulations. Contact details, including the dealer’s mobile number, are also captured to facilitate communication and authentication throughout the onboarding process.\n\nThe Dealer Profile module plays a critical role in 

## Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap=30)
split_docs = text_splitter.split_documents(docs)
split_docs

## Text Embedding & Vector Store Initialization

In [21]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


## Retrieve Relevant Chunks

In [18]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents = split_docs, embedding = embeddings)

retriever = vectorstore.as_retriever()

query = 'what is this document about ?'
response = retriever.invoke(query)
print(response[0].page_content)

This document is about the workflow of dealer on-boarding taking place in Daimler Trucks


## Integrate LLM 

In [27]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-5')

def askQuestion(query):
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""
    Answer the question based only on the context below.
    If the answer is not in the context, say: I don't know.
    
    Context:
    {context}
    
    Question:
    {query}
    """
    response = llm.invoke(prompt)
    print(response.content)

askQuestion('explain the steps')
askQuestion('what is VIP Program')

- Initiate the dealer onboarding process.
- Create the dealer’s basic profile.
- Once the basic profile is successfully created, transition the dealer to the Electronic Fund Transfer (EFT) stage.
- Proceed to downstream processes, which include calculating incentives and ensuring all program rules are consistently applied.
The VIP Program is the Volume Incentive Program. It’s designed to provide incentives to dealers, with clearly defined eligibility and ineligibility criteria, and is managed through a VIP module that tracks dealer performance and calculates incentives.
